# Deep Neural Decision Tree (DNDT) and Forest (DNDF) Reliability Study
### End-to-End Google Colab Runner for COVID-RARS

**Reference Paper:**  
Rofiqul Islam, Nihad Karim Chowdhury, and Muhammad Ashad Kabir. *"Robust COVID-19 detection from cough sounds using deep neural decision tree and forest: A comprehensive cross-datasets evaluation."* Expert Systems with Applications, Vol. 310, 2026, 131235. [DOI: 10.1016/j.eswa.2026.131235](https://doi.org/10.1016/j.eswa.2026.131235)

**Scientific Objective:**  
Evaluate whether differentiable DNDT/DNDF models sustain strong internal discrimination on Coswara respiratory audio across cough, speech, and breath, and test their stability under chronological drift and external transfer to COUGHVID.

## 1. Environment Setup & Colab GPU Check

In [ ]:
# Install dependencies if running on Google Colab
%pip install -q torch scikit-learn imbalanced-learn pandas numpy matplotlib seaborn

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Execution Device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive or Clone Repository

In [ ]:
import os
import sys
from pathlib import Path

# If running inside Colab, optionally mount Drive or set workspace root
try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_root = Path('/content/drive/MyDrive/Covid-RARS')
except Exception:
    repo_root = Path('.').resolve()

if repo_root.exists():
    sys.path.insert(0, str(repo_root / 'src'))
    print(f"Repo Root set to: {repo_root}")
else:
    print("Running in local repository mode.")
    sys.path.insert(0, str(Path('../src').resolve()))

## 3. Load Engineered Feature Banks

In [ ]:
from covid_rars.dndf_models import DNDFClassifier, NeuralDecisionTree, NeuralDecisionForest
from covid_rars.dndf_stages import run_dndf_reliability_pipeline
from covid_rars.dndf_reporting import build_dndf_summary_table

features_path = repo_root / "data/processed/features_compare_is10_top800.csv"
external_path = repo_root / "data/processed/features_compare_is10_coughvid_cough_top800.csv"

if not features_path.exists():
    # Fallback to demo synthetic data if raw features are not mounted in Colab
    print("Notice: Features file not found. Generating standardized benchmark feature frame for demonstration.")
    n_samples = 400
    n_feats = 100
    rng = np.random.RandomState(42)
    
    records = []
    for i in range(n_samples):
        p_id = f"p_{i:04d}"
        lbl = "positive" if rng.rand() > 0.65 else "negative"
        split = "train" if i < 280 else ("val" if i < 340 else "test")
        for mod in ["cough", "breath", "speech"]:
            row = {
                "participant_id": p_id,
                "recording_id": f"{p_id}_{mod}",
                "label_binary": lbl,
                "modality": mod,
                "split": split,
            }
            vec = rng.randn(n_feats) + (1.0 if lbl == "positive" else -0.5)
            for f_idx in range(n_feats):
                row[f"feat_{f_idx:03d}"] = vec[f_idx]
            records.append(row)
    features_df = pd.DataFrame(records)
    external_df = None
else:
    features_df = pd.read_csv(features_path)
    external_df = pd.read_csv(external_path) if external_path.exists() else None

print(f"Source Features Matrix Shape: {features_df.shape}")
if external_df is not None:
    print(f"External Features Matrix Shape: {external_df.shape}")

## 4. Run End-to-End DNDT & DNDF Reliability Pipeline

In [ ]:
artifacts = run_dndf_reliability_pipeline(
    features_df=features_df,
    external_features_df=external_df,
    modalities=["cough", "breath", "speech"],
    seeds=[1, 2, 5, 12, 40],
    num_trees=20,
    depth=4,
    used_features_rate=0.8,
    learning_rate=0.01,
    max_epochs=40,
    patience=8,
    use_smote=True,
    device=device,
    output_dir="reports/dndf_colab",
)

print("\n=== DNDT / DNDF FINAL SUMMARY TABLE ===")
display(artifacts.final_summary_table)

## 5. Visualizing Calibration and Reliability Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Summary Bar Chart of Track A vs Track B vs Track C
summary = artifacts.final_summary_table
if not summary.empty:
    axes[0].barh(summary["track"] + " : " + summary["modality"] + " (" + summary["model_name"] + ")", summary["mean_auroc"], color="teal", alpha=0.8)
    axes[0].set_xlim(0.4, 1.0)
    axes[0].set_xlabel("AUROC")
    axes[0].set_title("DNDT / DNDF Validation Ladder Performance")
    axes[0].grid(axis="x", linestyle="--", alpha=0.6)

# 2. Decision Curve Net Benefit
dca = artifacts.dca_summary
if not dca.empty:
    for key, grp in dca.groupby("model_name"):
        axes[1].plot(grp["threshold_probability"], grp["net_benefit_model"], label=f"{key}", lw=2)
    first_grp = list(dca.groupby("model_name"))[0][1]
    axes[1].plot(first_grp["threshold_probability"], first_grp["net_benefit_all"], label="Treat All", linestyle=":", color="gray")
    axes[1].axhline(0, color="black", linestyle="--", label="Treat None")
    axes[1].set_xlabel("Threshold Probability")
    axes[1].set_ylabel("Net Benefit")
    axes[1].set_title("Decision Curve Analysis (DCA)")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## 6. Exporting Results & Publication Artifacts

In [ ]:
print("All DNDT / DNDF artifacts and tables generated successfully:")
print(" - reports/dndf_colab/dndf_final_validation_summary.csv")
print(" - reports/dndf_colab/dndf_calibration_summary.csv")
print(" - reports/dndf_colab/dndf_operating_points.csv")
print(" - reports/dndf_colab/dndf_decision_curves.csv")
print(" - reports/dndf_colab/dndf_bootstrap_ci.csv")